In [1]:
print("a")

a


In [ ]:
import numpy as np
from scipy.special import erfc
from itertools import product
 
def Fhit_function(radius, distance, diffusionCoef, t):
    """Calculates the cumulative hitting probability up to time t."""
    if t <= 0:
        return 0.0
    return (radius / (distance + radius)) * erfc(distance / np.sqrt(4 * diffusionCoef * t))
 
def bit_error_rate(mem_len, threshold, N, radius, distance, diffusionCoef, Ts):
    """
    Calculates the Bit Error Rate (BER) accounting for Inter-Symbol Interference (ISI)
    using a Gaussian approximation.
   
    Parameters:
    - mem_len: Total length of the bit sequence to consider (current bit + previous bits)
    - threshold: The decision threshold (number of molecules)
    - N: Number of molecules released for bit '1'
    - radius: Radius of the receiver
    - distance: Distance between transmitter and receiver
    - diffusionCoef: Diffusion coefficient
    - Ts: Symbol duration (time allocated per bit)
    """
   
    # 1. Calculate interval hitting probabilities (Channel Coefficients)
    # P[0] is probability of hitting in the current slot [0, Ts]
    # P[1] is hitting in [Ts, 2Ts], representing ISI from the previous bit, etc.
    P = np.zeros(mem_len)
    for i in range(mem_len):
        t_end = (i + 1) * Ts
        t_start = i * Ts
        P[i] = Fhit_function(radius, distance, diffusionCoef, t_end) - Fhit_function(radius, distance, diffusionCoef, t_start)
       
    total_ber = 0.0
    num_sequences = 2**mem_len
   
    # 2. Generate all possible bit sequences of length mem_len
    sequences = list(product([0, 1], repeat=mem_len))
   
    for seq in sequences:
        # Reverse sequence so seq_rev[0] is the current bit, seq_rev[1] is the previous bit, etc.
        seq_rev = seq[::-1]
        current_bit = seq_rev[0]
       
        # 3. Calculate Mean and Variance for this specific sequence
        mu = 0.0
        variance = 0.0
        for i in range(mem_len):
            bit = seq_rev[i]
            mu += bit * N * P[i]
            # Variance of a sum of independent Binomials: np(1-p)
            variance += bit * N * P[i] * (1 - P[i])
           
        std = np.sqrt(variance)
       
        # 4. Calculate Conditional Error Probability using Gaussian Q-function equivalent
        # Q(x) = 0.5 * erfc(x / sqrt(2))
        if std == 0:
            # Edge case: If sequence is all zeros, there are no molecules, hence no variance.
            if current_bit == 1:
                pe = 1.0 if mu < threshold else 0.0
            else:
                pe = 1.0 if mu >= threshold else 0.0
        else:
            if current_bit == 1:
                # True bit is 1, error if received signal < threshold
                pe = 0.5 * erfc((mu - threshold) / (std * np.sqrt(2)))
            else:
                # True bit is 0, error if received signal > threshold
                pe = 0.5 * erfc((threshold - mu) / (std * np.sqrt(2)))
               
        total_ber += pe
       
    # 5. Average the error probabilities across all possible sequences
    average_ber = total_ber / num_sequences
   
    return average_ber
 
# --- Example Usage ---
# Assuming your physical parameters from the previous script:
p_radius = 5
p_distance = 12.5 - p_radius  # Distance from point source to receiver surface
p_diffusionCoef = 79.4
N_molecules = 10**6
 
# New digital communication parameters
symbol_duration = 1  # Ts in seconds
decision_threshold = 20000  # Arbitrary threshold for decoding
memory_length = 5  # 5-bit sequence (current bit + 4 previous bits)
 
ber = bit_error_rate(
    mem_len=memory_length,
    threshold=decision_threshold,
    N=N_molecules,
    radius=p_radius,
    distance=p_distance,
    diffusionCoef=p_diffusionCoef,
    Ts=symbol_duration
)
 
print(f"Calculated BER for memory length {memory_length}: {ber}")

In [ ]:
# Parameter lists for simulation sweep
import pandas as pd
from itertools import product

# Define parameter ranges
param_ranges = {
    'p_radius': [3, 5, 7],
    'p_distance': [7.5, 12.5, 15],  # Will calculate based on radius
    'p_diffusionCoef': [50, 79.4, 100],
    'N_molecules': [1e5, 1e6, 1e7],
    'symbol_duration': [0.5, 1, 2],
    'decision_threshold': [10000, 20000, 50000],
    'memory_length': [3, 5, 7]
}

# Create a list of parameter combinations
# For now, let's create a simpler approach: individual parameter lists
p_radius_list = [3, 5, 7]
p_diffusionCoef_list = [50, 79.4, 100]
N_molecules_list = [1e5, 1e6, 1e7]
symbol_duration_list = [0.5, 1, 2]
decision_threshold_list = [10000, 20000, 50000]
memory_length_list = [3, 5, 7]

# Create results storage
results = []

print("Parameter lists defined:")
print(f"  p_radius: {p_radius_list}")
print(f"  p_diffusionCoef: {p_diffusionCoef_list}")
print(f"  N_molecules: {N_molecules_list}")
print(f"  symbol_duration: {symbol_duration_list}")
print(f"  decision_threshold: {decision_threshold_list}")
print(f"  memory_length: {memory_length_list}")

In [ ]:
# Run simulations with parameter sweep
import time

total_simulations = (len(p_radius_list) * len(p_diffusionCoef_list) * 
                    len(N_molecules_list) * len(symbol_duration_list) * 
                    len(decision_threshold_list) * len(memory_length_list))

print(f"Total simulations to run: {total_simulations}")
print("Starting parameter sweep...")

start_time = time.time()
sim_count = 0

# Iterate through all parameter combinations
for radius in p_radius_list:
    for diffusion in p_diffusionCoef_list:
        for n_mol in N_molecules_list:
            for symbol_dur in symbol_duration_list:
                for threshold in decision_threshold_list:
                    for mem_len in memory_length_list:
                        # Calculate distance from radius
                        distance = 12.5 - radius
                        
                        # Run BER calculation
                        try:
                            ber = bit_error_rate(
                                mem_len=mem_len,
                                threshold=int(threshold),
                                N=int(n_mol),
                                radius=radius,
                                distance=distance,
                                diffusionCoef=diffusion,
                                Ts=symbol_dur
                            )
                            
                            # Store results
                            results.append({
                                'p_radius': radius,
                                'p_distance': distance,
                                'p_diffusionCoef': diffusion,
                                'N_molecules': int(n_mol),
                                'symbol_duration': symbol_dur,
                                'decision_threshold': int(threshold),
                                'memory_length': mem_len,
                                'BER': ber
                            })
                            
                            sim_count += 1
                            if sim_count % 10 == 0:
                                print(f"  Completed {sim_count}/{total_simulations} simulations...")
                        
                        except Exception as e:
                            print(f"Error at iteration {sim_count}: {e}")
                            continue

elapsed_time = time.time() - start_time
print(f"\nCompleted {sim_count} simulations in {elapsed_time:.2f} seconds")
print(f"Average time per simulation: {elapsed_time/sim_count:.4f} seconds")

In [ ]:
# Convert results to DataFrame and save
df_results = pd.DataFrame(results)

print(f"Results DataFrame shape: {df_results.shape}")
print("\nFirst few rows:")
print(df_results.head(10))

print("\nResults summary statistics:")
print(df_results[['BER']].describe())

# Save to CSV
output_file = 'ber_simulation_results.csv'
df_results.to_csv(output_file, index=False)
print(f"\nResults saved to: {output_file}")

In [ ]:
# Analysis and filtering examples

# Find the best (lowest) BER
best_result = df_results.loc[df_results['BER'].idxmin()]
print("Best Configuration (Lowest BER):")
print(best_result)
print(f"\nBER: {best_result['BER']:.6e}\n")

# Filter results by criteria
print("=" * 60)
print("Example filters:\n")

# Results with BER < 0.001
low_ber = df_results[df_results['BER'] < 0.001]
print(f"Configurations with BER < 0.001: {len(low_ber)}")

# Results for specific memory length
mem_len_5 = df_results[df_results['memory_length'] == 5]
print(f"Configurations with memory_length=5: {len(mem_len_5)}")

# Group by memory length and show average BER
print("\nAverage BER by Memory Length:")
print(df_results.groupby('memory_length')['BER'].agg(['mean', 'min', 'max']))

print("\nAverage BER by Diffusion Coefficient:")
print(df_results.groupby('p_diffusionCoef')['BER'].agg(['mean', 'min', 'max']))

print("\nAverage BER by Decision Threshold:")
print(df_results.groupby('decision_threshold')['BER'].agg(['mean', 'min', 'max']))

In [ ]:
import plotly.graph_objects as go
import numpy as np

# 1. Define fixed parameters (using your previous example values)
fixed_params = {
    'mem_len': 5,
    'N': int(10**6),
    'radius': 5,
    'distance': 7.5, # 12.5 - 5
    'diffusionCoef': 79.4,
    'Ts': 1
}

# 2. Define the threshold sweep range
thresholds_sweep = np.linspace(70000, 500000, 1000) # Change to see different threshold and ranges
ber_values = []

# 3. Calculate BER for each threshold
print("Calculating BER values for the sweep...")
for th in thresholds_sweep:
    ber = bit_error_rate(threshold=th, **fixed_params)
    ber_values.append(ber)

# 4. Find the empirical minimum point from our sweep for highlighting
min_idx = np.argmin(ber_values)
optimal_th = thresholds_sweep[min_idx]
min_ber = ber_values[min_idx]

# 5. Create the Plotly figure
fig = go.Figure()

# Add the main BER curve
fig.add_trace(go.Scatter(
    x=thresholds_sweep, 
    y=ber_values,
    mode='lines',
    name='BER',
    line=dict(color='royalblue', width=3)
))

# Add a prominent marker for the minimum BER
fig.add_trace(go.Scatter(
    x=[optimal_th], 
    y=[min_ber],
    mode='markers+text',
    name='Minimum',
    marker=dict(color='red', size=12, symbol='x'),
    text=[f'Min BER: {min_ber:.2e}<br>λ*: {optimal_th:.0f}'],
    textposition='top center',
    textfont=dict(color='red')
))

# Update layout for log scale, labels, and clean styling
fig.update_layout(
    title='Bit Error Rate (BER) vs. Decision Threshold (λ)',
    xaxis_title='Decision Threshold (λ) [Number of Molecules]',
    yaxis_title='Bit Error Rate (BER)',
    yaxis_type='log',  # Crucial for visualizing BER
    template='plotly_white',
    hovermode='x unified', # Shows a clean hover tooltip for both x and y
    showlegend=False,
    width=900,
    height=600
)

# Render the plot
fig.show()